In [1]:

from mpramnist.Rafi2024 import RafiDataset
from mpramnist.Rafi2024 import LitModel_Rafi

# `transforms` (t)        -> sequence-level augmentations (flanks, cropping, RC, one-hot, ...)
# `joint_transforms` (j_t) -> transforms that need BOTH the sequence and the raw target
#                              at the same time (e.g. IsSingleton below), applied before
#                              the target itself is converted to soft-bin probabilities
# `target_transforms` (t_t)-> transforms that turn the raw scalar activity label into the
#                              soft classification target (SoftBinTarget)
import mpramnist.transforms as t
import mpramnist.joint_transforms as j_t
import mpramnist.target_transforms as t_t

from mpramnist.models import HumanLegNet
from mpramnist.models import initialize_weights

import torch
import torch.nn as nn
import torch.utils.data as data
from torchmetrics import PearsonCorrCoef

import lightning.pytorch as L
import torch.nn.functional as F

BATCH_SIZE = 1024
NUM_WORKERS = 16

length = 120
plasmid = RafiDataset.PLASMID.upper()
insert_start = plasmid.find("N" * 80)
right_flank = RafiDataset.RIGHT_FLANK
left_flank = plasmid[insert_start - length : insert_start]

print(RafiDataset.TYPES)

['all', 'high', 'low', 'yeast', 'random', 'challenging', 'snv', 'perturbation', 'tiling']


/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Soft classification target and the singleton flag

**How the activity label is produced in the experiment.** Cells are separated with a
FACS (cell sorter) into a fixed number of bins according to the ratio of the two
reporter proteins' fluorescence. For every sequence we count how many reads/cells
carrying it landed in each bin. The activity of a sequence is the **weighted average
of the bin indices**, weighted by how many reads of that sequence fell into each bin.

**What this notebook does with that.**
1. The model predicts a distribution over the `n_bins = 18` FACS bins (a classification
   head) instead of a single scalar.
2. `t_t.SoftBinTarget()` converts each sequence's continuous weighted-average label into
   a *soft probability vector* over the 18 bins (assuming a small amount of Gaussian
   measurement noise around the label), so the classification target mirrors how the
   label was actually generated.
3. `j_t.IsSingleton()` / `j_t.NotASingleton()` add an extra input feature channel that
   tells the model whether a sequence's label is a "singleton" (all of its reads came
   from exactly one bin, so the label is an exact integer and therefore more reliable)
   or a mixture across bins (a non-integer weighted average, i.e. noisier evidence).
4. At inference time the predicted bin-probabilities are collapsed back into a single
   continuous score by taking their expectation over bin indices (`Rafi_Probs` /
   `AutosomeFinalLayersBlock` below), so downstream code (Pearson correlation, etc.)
   works with a scalar prediction.


In [2]:
class AutosomeFinalLayersBlock(nn.Module):
    """Turns the model's raw 18-way logits into (a) a continuous activity score and
    (b) log-probabilities usable for the soft-target loss.

    - `score` is the *expected bin index* under the predicted distribution, i.e.
      sum_k( softmax(logits)_k * k ). This converts "probability of landing in each
      FACS bin" into a single continuous activity value on the same 0..17 scale that
      SoftBinTarget's soft labels live on.
    - During training we also need `logprobs` (log_softmax) so the loss can compare
      them against the soft-bin target distribution produced by SoftBinTarget.
    - During eval/inference we only care about the scalar score, so a bare tensor is
      returned instead of the dict.
    """
    def __init__(
        self,
    ):
        super().__init__()
        out_channels=18

        # bin indices 0..17, used as weights when computing the expected value
        self.register_buffer('bins', torch.arange(start=0, end=out_channels, step=1, requires_grad=False))

    def forward(self, x):
        logprobs = F.log_softmax(x, dim=1) 
        x = F.softmax(x, dim=1)
        # expected activity = weighted average of bin indices under the predicted distribution
        score = (x * self.bins).sum(dim=1)
        if self.training:
            # return both the scalar estimate and the log-probs, so the loss can be
            # computed against SoftBinTarget's soft probability vector
            return {'value': score, 'probs': logprobs}
        else:
            # at inference time we only need the scalar activity estimate
            return score

class Rafi_Probs(nn.Module):
    """
     Backbone (HumanLegNet or another model) -> 18-way classification head (AutosomeFinalLayersBlock),
    so the rest of the training/eval code can treat the whole thing as a single model.
    """
    def __init__(self, model):
        super().__init__()
        
        self.model = model
        self.prob_and_value = AutosomeFinalLayersBlock()

    def forward(self, x):
        x = self.model(x)
        x = self.prob_and_value(x)
        return x

### Preprocessing

- `t.AddFeatureChannels(channels=['is_singleton'])` appends the `is_singleton` flag
  (set below by the joint/target transforms) as an extra input channel to the
  one-hot-encoded sequence, so the model can learn to trust singleton labels
  (exact bin, less noisy) more than mixed-bin labels.
- `j_t.IsSingleton()` (train/val) looks at the raw target *before* it is converted to
  soft-bin probabilities and sets `is_singleton = 1` if the label is an integer (i.e.
  all reads for that sequence came from a single FACS bin), otherwise `0`. Test data
  has no per-read/group breakdown available at this stage, so we don't know the
  singleton status; `j_t.NotASingleton()` simply hardcodes `is_singleton = 0` for every
  test sequence so the extra channel still has a defined value.


In [3]:
# preprocessing
train_transform = t.Compose([t.AddFlanks(left_flank, right_flank),
                             t.LeftCrop(length, length),
                             t.ReverseComplement(0.5), 
                             t.AddFeatureChannels(channels=['is_singleton']), # add the singleton flag as an extra input channel    
                             t.AddReverseChannel(),
                             t.Seq2Tensor()])

val_transform = t.Compose([t.AddFlanks(left_flank, right_flank),
                                t.LeftCrop(length, length),
                                t.ReverseComplement(0),
                                t.AddFeatureChannels(channels=['is_singleton']), # same channel, computed from the val target below
                                t.AddReverseChannel(),
                                t.Seq2Tensor(),])

test_transform = t.Compose([t.AddFlanks(left_flank, right_flank),
                                t.LeftCrop(length, length),
                                t.ReverseComplement(0),
                                j_t.NotASingleton(), # test targets aren't on the bin scale, so we can't tell singleton status -> always 0
                                t.AddFeatureChannels(channels=['is_singleton']), 
                                t.AddReverseChannel(),
                                t.Seq2Tensor(),])

In [4]:
# load the data
train_dataset = RafiDataset(split="train", transform=train_transform,
                            target_transform=t_t.SoftBinTarget(), # raw label -> soft distribution over 18 bins
                            joint_transform=j_t.IsSingleton(), root="../data")
val_dataset = RafiDataset(split="val", data_type=["all"], transform=val_transform,
                          joint_transform=j_t.IsSingleton(), # val labels are also on the bin scale -> derive singleton flag
                            root="../data")
test_dataset = RafiDataset(split="test", data_type=["all"], transform=test_transform, root="../data") # no target_transform: evaluated on the raw continuous score

print(len(train_dataset), len(val_dataset), len(test_dataset))

# encapsulate data into dataloader form
train_loader = data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = data.DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = data.DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

in_channels = len(train_dataset[0][0])

/home/nios/MPRA-MNIST/mpramnist/data/Dream/Dream_train.tsv
/home/nios/MPRA-MNIST/mpramnist/data/Dream/Dream_single.tsv
/home/nios/MPRA-MNIST/mpramnist/data/Dream/Dream_single.tsv
6739258 9045 62058


In [5]:
out_channels = 18  # one output per FACS bin (classification head)
legnet = HumanLegNet(
        in_ch=in_channels,
        output_dim=out_channels,
        stem_ch=64,
        stem_ks=7,
        ef_ks=7,
        ef_block_sizes=[256, 128, 128, 64, 64, 64, 64],
        pool_sizes=[1, 1, 1, 1, 1, 1, 1],
        resize_factor=4,
    )
legnet.apply(initialize_weights)

# wrap the backbone with the classification head that also produces a scalar
# activity estimate (expected bin index)
model = Rafi_Probs(legnet)

seq_model = LitModel_Rafi(model=model, weight_decay=1e-2, lr=1e-2, print_each=1)

# Initialize a trainer
trainer = L.Trainer(
    accelerator="gpu",
    devices=[0],
    max_epochs=1,
    gradient_clip_val=1,
    precision="16-mixed",
    enable_progress_bar=True,
    num_sanity_val_steps=0,
    enable_model_summary=False
)
# Train the model
trainer.fit(seq_model, train_dataloaders=train_loader, val_dataloaders=val_loader)
trainer.test(seq_model, dataloaders=test_loader)

Using 16bit Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.
/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isi

Epoch 0: 100%|██████████| 6582/6582 [41:23<00:00,  2.65it/s, v_num=2]
---------------------------------------------------------------------------------
| Epoch: 0 | Val Loss: 139.98630 | Val Pearson: 0.95748 | Train Pearson: 0.71827 
---------------------------------------------------------------------------------

Epoch 0: 100%|██████████| 6582/6582 [41:26<00:00,  2.65it/s, v_num=2, val_loss=140.0, val_pearson=0.957, train_loss=1.060]

Metric val_loss improved. New best score: 139.986
`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████| 6582/6582 [41:26<00:00,  2.65it/s, v_num=2, val_loss=140.0, val_pearson=0.957, train_loss=1.060]


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Testing DataLoader 0: 100%|██████████| 61/61 [00:06<00:00,  9.56it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss           137.52857971191406
      test_pearson          0.9496899843215942
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 137.52857971191406, 'test_pearson': 0.9496899843215942}]

## Evaluating Single sequences

In [8]:
forw_transform = t.Compose([t.AddFlanks(left_flank, right_flank),
                                t.LeftCrop(length, length),
                                t.ReverseComplement(0),
                                j_t.NotASingleton(),
                                t.AddFeatureChannels(channels=['is_singleton']),
                                t.AddReverseChannel(),
                                t.Seq2Tensor(),])
rev_transform = t.Compose([t.AddFlanks(left_flank, right_flank),
                                t.LeftCrop(length, length),
                                t.ReverseComplement(1),
                                j_t.NotASingleton(),
                                t.AddFeatureChannels(channels=['is_singleton']),
                                t.AddReverseChannel(),
                                t.Seq2Tensor(),])
def meaned_prediction(forw, rev, trainer, seq_model, name, is_paired=False):
    predictions_forw = trainer.predict(seq_model, dataloaders=forw)
    targets = torch.cat([pred["target"] for pred in predictions_forw])
    y_preds_forw = torch.cat([pred["ref_predicted"] for pred in predictions_forw])

    predictions_rev = trainer.predict(seq_model, dataloaders=rev)
    y_preds_rev = torch.cat([pred["ref_predicted"] for pred in predictions_rev])

    mean_forw = torch.mean(torch.stack([y_preds_forw, y_preds_rev]), dim=0)

    pears = PearsonCorrCoef()
    print("Task '" + name + "' Pearson r^2")

    if is_paired:
        y_preds_forw_alt = torch.cat(
            [pred["alt_predicted"] for pred in predictions_forw]
        )
        y_preds_rev_alt = torch.cat([pred["alt_predicted"] for pred in predictions_rev])
        mean_alt = torch.mean(torch.stack([y_preds_forw_alt, y_preds_rev_alt]), dim=0)
        pred = mean_alt - mean_forw
        return pears(pred, targets) * pears(pred, targets)

    return pears(mean_forw, targets) * pears(mean_forw, targets)

### All Sequences

In [9]:
test_forw = RafiDataset(split="test", data_type=["all"], transform=forw_transform, root="../data")
test_rev = RafiDataset(split="test", data_type=["all"], transform=rev_transform, root="../data")

forw = data.DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev = data.DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)

meaned_prediction(forw, rev, trainer, seq_model, "All Sequences")

/home/nios/MPRA-MNIST/mpramnist/data/Dream/Dream_single.tsv
/home/nios/MPRA-MNIST/mpramnist/data/Dream/Dream_single.tsv


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting DataLoader 0:   0%|          | 0/61 [09:23<?, ?it/s]
Predicting: |          | 61/? [00:02<00:00, 28.08it/s]


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 61/61 [00:01<00:00, 31.90it/s]
Task 'All Sequences' Pearson r^2


tensor(0.9060)

### High

In [10]:
test_forw = RafiDataset(split="test", data_type=["high"], transform=forw_transform, root="../data")
test_rev = RafiDataset(split="test", data_type=["high"], transform=rev_transform, root="../data")

forw = data.DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev = data.DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)


meaned_prediction(forw, rev, trainer, seq_model, "High")

/home/nios/MPRA-MNIST/mpramnist/data/Dream/Dream_single.tsv
/home/nios/MPRA-MNIST/mpramnist/data/Dream/Dream_single.tsv


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00,  1.68it/s]

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]



Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 31.24it/s]
Task 'High' Pearson r^2


tensor(0.3867)

### Low

In [11]:
test_forw = RafiDataset(split="test", data_type=["low"], transform=forw_transform, root="../data")
test_rev = RafiDataset(split="test", data_type=["low"], transform=rev_transform, root="../data")

forw = data.DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev = data.DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)


meaned_prediction(forw, rev, trainer, seq_model, "Low")

/home/nios/MPRA-MNIST/mpramnist/data/Dream/Dream_single.tsv
/home/nios/MPRA-MNIST/mpramnist/data/Dream/Dream_single.tsv


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00,  1.78it/s]

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]



Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 26.97it/s]
Task 'Low' Pearson r^2


tensor(0.3796)

### Native

In [12]:
test_forw = RafiDataset(split="test", data_type=["yeast"], transform=forw_transform, root="../data")
test_rev = RafiDataset(split="test", data_type=["yeast"], transform=rev_transform, root="../data")

forw = data.DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev = data.DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)


meaned_prediction(forw, rev, trainer, seq_model, "Native")

/home/nios/MPRA-MNIST/mpramnist/data/Dream/Dream_single.tsv
/home/nios/MPRA-MNIST/mpramnist/data/Dream/Dream_single.tsv


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00,  1.62it/s]

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]



Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 38.70it/s]
Task 'Native' Pearson r^2


tensor(0.6935)

### Random

In [13]:
test_forw = RafiDataset(split="test", data_type=["random"], transform=forw_transform, root="../data")
test_rev = RafiDataset(split="test", data_type=["random"], transform=rev_transform, root="../data")

forw = data.DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev = data.DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)


meaned_prediction(forw, rev, trainer, seq_model, "Random")

/home/nios/MPRA-MNIST/mpramnist/data/Dream/Dream_single.tsv
/home/nios/MPRA-MNIST/mpramnist/data/Dream/Dream_single.tsv


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting DataLoader 0: 100%|██████████| 6/6 [00:00<00:00,  8.36it/s]

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]



Predicting DataLoader 0: 100%|██████████| 6/6 [00:00<00:00, 32.97it/s]
Task 'Random' Pearson r^2


tensor(0.9405)

### Challenging

In [14]:
test_forw = RafiDataset(split="test", data_type=["challenging"], transform=forw_transform, root="../data")
test_rev = RafiDataset(split="test", data_type=["challenging"], transform=rev_transform, root="../data")

forw = data.DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev = data.DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)


meaned_prediction(forw, rev, trainer, seq_model, "Challenging")

/home/nios/MPRA-MNIST/mpramnist/data/Dream/Dream_single.tsv
/home/nios/MPRA-MNIST/mpramnist/data/Dream/Dream_single.tsv


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting DataLoader 0: 100%|██████████| 2/2 [00:00<00:00,  3.40it/s]

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]



Predicting DataLoader 0: 100%|██████████| 2/2 [00:00<00:00, 32.40it/s]
Task 'Challenging' Pearson r^2


tensor(0.8548)

## Evaluating Paired sequences

### SNVs

In [15]:
test_forw = RafiDataset(split="test", data_type=["snv"], transform=forw_transform, root="../data")
test_rev = RafiDataset(split="test", data_type=["snv"], transform=rev_transform, root="../data")

forw = data.DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev = data.DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)


meaned_prediction(forw, rev, trainer, seq_model, "SNVs", is_paired=True)

/home/nios/MPRA-MNIST/mpramnist/data/Dream/Dream_paired.tsv
/home/nios/MPRA-MNIST/mpramnist/data/Dream/Dream_paired.tsv


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting DataLoader 0: 100%|██████████| 39/39 [00:02<00:00, 13.19it/s]


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 39/39 [00:02<00:00, 16.10it/s]
Task 'SNVs' Pearson r^2


tensor(0.6740)

### Motif Perturbation

In [16]:
test_forw = RafiDataset(split="test", data_type=["perturbation"], transform=forw_transform, root="../data")
test_rev = RafiDataset(split="test", data_type=["perturbation"], transform=rev_transform, root="../data")

forw = data.DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev = data.DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)


meaned_prediction(forw, rev, trainer, seq_model, "Motif Perturbation", is_paired=True)

/home/nios/MPRA-MNIST/mpramnist/data/Dream/Dream_paired.tsv
/home/nios/MPRA-MNIST/mpramnist/data/Dream/Dream_paired.tsv


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting DataLoader 0: 100%|██████████| 3/3 [00:00<00:00,  4.16it/s]

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]



Predicting DataLoader 0: 100%|██████████| 3/3 [00:00<00:00, 15.55it/s]
Task 'Motif Perturbation' Pearson r^2


tensor(0.9280)

### Motif Tiling

In [17]:
test_forw = RafiDataset(split="test", data_type=["tiling"], transform=forw_transform, root="../data")
test_rev = RafiDataset(split="test", data_type=["tiling"], transform=rev_transform, root="../data")

forw = data.DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev = data.DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)


meaned_prediction(forw, rev, trainer, seq_model, "Motif Tiling", is_paired=True)

/home/nios/MPRA-MNIST/mpramnist/data/Dream/Dream_paired.tsv
/home/nios/MPRA-MNIST/mpramnist/data/Dream/Dream_paired.tsv


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting DataLoader 0: 100%|██████████| 3/3 [00:00<00:00,  4.25it/s]


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 3/3 [00:00<00:00, 19.66it/s]
Task 'Motif Tiling' Pearson r^2


tensor(0.8233)